# Thai Gender-Bias BERTopic Pipeline (Colab-ready)

End-to-end notebook combining preprocessing, modeling, postprocessing, and bias-topic extraction.

## What this notebook does

- Installs deps
- Preprocesses Thai social text
- Fits BERTopic (with SentenceTransformer embeddings)
- Reduces/cleans topics, detects gender/sexism-related topics
- Samples example sentences for annotation
- Saves outputs and the model

## Notes

- For offline use, upload your SentenceTransformer model folder to Colab and point `EMBEDDING_MODEL_PATH` to it.
- For online use, it will download `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` by default.
- Expected input: CSV with a `text` column (you can change `TEXT_COLUMN`).


In [ ]:
# %%capture
# !pip install -U pip
# !pip install bertopic pythainlp sentence-transformers umap-learn hdbscan scikit-learn pandas

: 

In [ ]:
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from pythainlp.corpus import thai_stopwords
from pythainlp.tokenize import word_tokenize

# os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

: 

In [ ]:
# Preprocessing utilities
STOPWORDS = set(thai_stopwords())

def clean_text(text: str) -> str:
    import re

    text = "" if text is None else str(text)
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"@[A-Za-z0-9_]+", "", text)
    text = re.sub(r"#\S+", "", text)

    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols
        "\U0001F680-\U0001F6FF"  # transport
        "\U0001F1E0-\U0001F1FF"  # flags
        "]+",
        flags=re.UNICODE,
    )
    text = emoji_pattern.sub("", text)
    return text.strip()


def thai_tokenize(text: str, stopwords: Iterable[str] | None = None) -> str:
    tokens = word_tokenize(text, engine="newmm")
    stopwords = STOPWORDS if stopwords is None else set(stopwords)
    tokens = [token for token in tokens if token and token not in stopwords]
    return " ".join(tokens)


def preprocess(text: str, stopwords: Iterable[str] | None = None) -> str:
    return thai_tokenize(clean_text(text), stopwords=stopwords)

In [ ]:
# Pipeline configuration and helpers
DEFAULT_BIAS_KEYWORDS: Tuple[str, ...] = (
    "ผู้หญิง",
    "ผู้ชาย",
    "เมีย",
    "ผัว",
    "ตุ๊ด",
    "เกย์",
    "กะเทย",
    "สาว",
    "อี",
    "มัน",
    "อารมณ์",
    "งอแง",
    "อ่อนแอ",
    "แรง",
    "หึง",
    "เจ้าชู้",
    "ทำงาน",
    "ทำอาหาร",
    "หาเงิน",
)
DEFAULT_EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


@dataclass
class TopicPipelineConfig:
    embedding_model: str = DEFAULT_EMBEDDING_MODEL
    min_df: int = 3
    ngram_range: Tuple[int, int] = (1, 2)
    min_topic_size: int = 30
    nr_topics: int | str | None = "auto"
    reduce_to: int | None = 25
    sample_size: int = 200
    seed: int = 42


@dataclass
class PipelineResult:
    topic_model: BERTopic
    topics: List[int]
    probabilities: List[List[float]] | None
    df: pd.DataFrame
    clean_df: pd.DataFrame
    bias_topics: List[int] = field(default_factory=list)
    biased_samples: Dict[int, List[str]] = field(default_factory=dict)

    def save(self, output_csv: str, model_dir: str | None = None) -> None:
        self.df.to_csv(output_csv, index=False)
        if model_dir:
            self.topic_model.save(model_dir)


def build_topic_model(config: TopicPipelineConfig) -> BERTopic:
    embedding_model = SentenceTransformer(config.embedding_model)
    vectorizer = CountVectorizer(
        ngram_range=config.ngram_range,
        min_df=config.min_df,
    )
    return BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer,
        language="thai",
        calculate_probabilities=True,
        min_topic_size=config.min_topic_size,
        nr_topics=config.nr_topics,
    )


def detect_bias_topics(topic_model: BERTopic, bias_keywords: Iterable[str]) -> List[int]:
    bias_keywords = list(bias_keywords)
    matched_topics: List[int] = []
    for topic_id, words in topic_model.get_topics().items():
        if topic_id == -1:
            continue
        word_string = " ".join([word for word, _ in words])
        if any(keyword in word_string for keyword in bias_keywords):
            matched_topics.append(topic_id)
    return matched_topics


def extract_samples(df: pd.DataFrame, topic_id: int, n: int, seed: int) -> List[str]:
    topic_df = df[df["topic"] == topic_id]
    if topic_df.empty:
        return []
    sampled = topic_df.sample(min(n, len(topic_df)), random_state=seed)
    return sampled["text"].astype(str).tolist()


def run_pipeline(
    df: pd.DataFrame,
    text_column: str = "text",
    bias_keywords: Sequence[str] | None = None,
    config: TopicPipelineConfig | None = None,
) -> PipelineResult:
    if config is None:
        config = TopicPipelineConfig()
    if bias_keywords is None:
        bias_keywords = DEFAULT_BIAS_KEYWORDS

    if text_column not in df.columns:
        raise ValueError(f"Missing text column '{text_column}' in dataframe")

    working_df = df.copy()
    working_df["clean"] = working_df[text_column].astype(str).apply(preprocess)
    texts = working_df["clean"].tolist()

    topic_model = build_topic_model(config)
    topics, probabilities = topic_model.fit_transform(texts)

    if config.reduce_to:
        topic_model = topic_model.reduce_topics(
            texts,
            topics=topics,
            probabilities=probabilities,
            nr_topics=config.reduce_to,
        )
        probabilities = topic_model.probabilities_

    doc_info = topic_model.get_document_info(texts)
    working_df["topic"] = doc_info["Topic"].tolist()
    clean_df = working_df[working_df["topic"] != -1].copy()

    bias_topics = detect_bias_topics(topic_model, bias_keywords)
    biased_samples = {
        topic_id: extract_samples(
            clean_df, topic_id, n=config.sample_size, seed=config.seed
        )
        for topic_id in bias_topics
    }

    return PipelineResult(
        topic_model=topic_model,
        topics=working_df["topic"].tolist(),
        probabilities=probabilities,
        df=working_df,
        clean_df=clean_df,
        bias_topics=bias_topics,
        biased_samples=biased_samples,
    )

## Load data

- Upload a CSV or mount Drive.
- Set `DATA_PATH` and `TEXT_COLUMN` below.


In [ ]:
DATA_PATH = "./assets/postprocessed_output.csv"  # change to your CSV
TEXT_COLUMN = "text"

df = pd.read_csv(DATA_PATH)
display(df.head())

## Configure embedding model

- If offline: upload a SentenceTransformer model folder and point `EMBEDDING_MODEL_PATH` to it.
- If online: leave as default to download automatically.


In [ ]:
# Set to a local path (preferred for offline) or a HF model id (online)
EMBEDDING_MODEL_PATH = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  # or "/content/drive/.../model_folder"

config = TopicPipelineConfig(
    embedding_model=EMBEDDING_MODEL_PATH,
    min_topic_size=30,
    reduce_to=25,
)
config

## Run pipeline


In [ ]:
result = run_pipeline(df, text_column=TEXT_COLUMN, config=config)
result.df.head()

## Bias topics and samples


In [ ]:
print("Bias topics:", result.bias_topics)

if result.bias_topics:
    first = result.bias_topics[0]
    print("\nTop words in first bias topic:")
    print(result.topic_model.get_topic(first))
    print("\nSampled sentences:")
    for line in result.biased_samples.get(first, [])[:10]:
        print("-", line)

## Save outputs

- `bertopic_output.csv` contains document-level topics.
- `bertopic_thai_gender_bias` stores the trained model.


In [ ]:
OUTPUT_CSV = "bertopic_output.csv"
MODEL_DIR = "bertopic_thai_gender_bias"

result.save(OUTPUT_CSV, model_dir=MODEL_DIR)
print(f"Saved CSV to {OUTPUT_CSV} and model to {MODEL_DIR}")